# TP2 NLP - Executor Colab

Este notebook apenas orquestra comandos do repositório. A implementação fica em `src`, `custom_metrics` e `scripts`.

## 1. Clonar ou entrar no repositório

Execute a célula de clone apenas quando o repositório ainda não estiver disponível no Colab.

In [ ]:
!git clone https://github.com/mh131105/TP2_NLP.git
%cd TP2_NLP
!pwd

In [ ]:
!git pull origin main

## 2. Instalar dependências

In [ ]:
!git pull origin main
!grep torch requirements.txt || echo "torch nao esta no requirements"
!pip install -r requirements.txt

## 3. Login opcional no Hugging Face

Defina `HF_TOKEN` nos segredos do Colab ou ajuste a célula de login.

In [ ]:
from huggingface_hub import login
import os
login(token=os.environ.get('HF_TOKEN'))

## 4. Preparar datasets

O `prepare_spider` primeiro usa `data/raw/spider` se a pasta já existir. Se ela não existir, o script pode importar um diretório ou ZIP/TAR passado com `--source_path`, ou tentar usar a fonte configurada no Hugging Face.

In [ ]:
!python -m scripts.prepare_spider --data_dir data/raw/spider --output_dir data/processed/spider
# If the automatic source fails, put a Spider ZIP/folder on Drive and use:
# !python -m scripts.prepare_spider --data_dir data/raw/spider --output_dir data/processed/spider --source_path /content/drive/MyDrive/spider.zip --force_download
!python -m scripts.prepare_mmlu --config configs/eval.yaml

## 5. Benchmark do baseline

In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/base

In [ ]:
!python -m scripts.evaluate_mmlu --config configs/eval.yaml --model_path outputs/base --output_dir outputs/base

## 6. Treinar experimentos LoRA

In [ ]:
!python -m scripts.train --config configs/train_lora_exp_a.yaml

In [ ]:
!python -m scripts.train --config configs/train_lora_exp_b.yaml

In [ ]:
!python -m scripts.train --config configs/train_lora_exp_c.yaml

## 7. Benchmarks finais

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_a

In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_b

In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_c

## 8. Smoke mode sem baixar modelos

Use isto apenas para validar o encadeamento dos scripts, não para reportar resultados.

In [ ]:
# !python -m pytest
# !python -m scripts.prepare_mmlu --config configs/eval.yaml --mock --limit_per_category 2
# !python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/base --mock --limit 2